# ECHO-Dialysis Study — Broad Echo Screening (all parameters <20% missing)

**File:** `echo_stage5_broad_screen_lt20.ipynb`

## Purpose
Structured screening of **19 pre-approved echocardiographic parameters** to identify:
1. Association with 1-year mortality and hospitalisation burden
2. Which parameters add prognostic information beyond EF
3. Overlap between related parameters within the same clinical domain
4. A prioritisation summary suitable for cardiologist review

**This is an exploratory broad screen — not the final model-selection step.**

## Sections
1. [Setup, variable registry, and data loading](#s1)
2. [Univariable mortality screening](#s2)
3. [Univariable hospitalisation screening](#s3)
4. [Adjusted mortality models](#s4)
5. [Adjusted hospitalisation models](#s5)
6. [Beyond-EF: matched-sample incremental models](#s6)
7. [Overlap analysis](#s7)
8. [Prioritisation summary](#s8)
9. [QA](#s9)

---
## 1. Setup, variable registry, and data loading <a id='s1'></a>

In [ ]:
import subprocess, sys
for pkg in ['statsmodels']:
    try: __import__(pkg)
    except ImportError: subprocess.check_call([sys.executable,'-m','pip','install',pkg,'-q'])

import pandas as pd
import numpy as np
import os
import warnings
warnings.filterwarnings('ignore')
import statsmodels.api as sm
import statsmodels.formula.api as smf
from scipy import stats

pd.set_option('display.max_columns', 30)
pd.set_option('display.float_format', '{:.4f}'.format)

# ── Constants ─────────────────────────────────────────────────
ONE_YEAR      = 365
RENAL_PRIMARY = 'creatinine-numeric result'   # change to 'GFR' if preferred
CI_W          = 10.0   # CI width threshold for unstable flag
COEF_T        = 5.0    # coefficient magnitude threshold
P_SCREEN      = 0.10   # univariable screening threshold

def safe(v): return v.replace('-','_').replace(' ','_').replace('/','_').replace('.','_')

def effective_miss(s, df_col):
    t = df_col.astype(str)
    return (df_col.isna() | t.eq('No Value') | t.eq('nan')).mean()

def get_series(df_m, col):
    df_m = df_m.loc[:, ~df_m.columns.duplicated()].copy()
    s = df_m[col]
    if isinstance(s, pd.DataFrame): s = s.iloc[:, 0]
    return s

# ── QA ────────────────────────────────────────────────────────
QA_LOG = []
def qa(cond, label, expected=None, actual=None):
    d = f' (exp={expected}, got={actual})' if expected is not None else ''
    msg = ('OK    ' if cond else 'WARN  ') + label + d
    QA_LOG.append(msg); print(msg)
    return cond

# ══════════════════════════════════════════════════════════════
# PRE-APPROVED VARIABLE REGISTRY
# Exact list from study protocol — do not modify without clinical review
# ══════════════════════════════════════════════════════════════
ECHO_REGISTRY = [
    # raw_name, domain, var_type, clinical_ref (None = continuous)

    # Diastolic / filling pressures
    ('TissueDopplerEERatioSeptal',             'diastolic/filling',          'continuous', None),
    ('TissueDopplerEERatioLateral',            'diastolic/filling',          'continuous', None),
    ('TissueDopplerEVelositySeptal',           'diastolic/filling',          'continuous', None),
    ('TissueDopplerEVelosityLateral',          'diastolic/filling',          'continuous', None),
    ('MitralInflowPeakEWave',                  'diastolic/filling',          'continuous', None),
    ('LACavitySize',                           'diastolic/filling',          'categorical','Normal'),

    # Pulmonary / right-sided / valvular
    ('EstimatedSysPAPressure',                 'pulmonary/right-sided load', 'continuous', None),
    ('MitralRegurgitation',                    'valvular',                   'categorical','Trivial'),
    ('TricuspidRegurgitation',                 'valvular/pulmonary-right',   'categorical','Trivial'),

    # LV systolic
    ('LV_EF',                                  'systolic',                   'continuous', None),

    # LV structure / remodelling
    ('LeftVentricleEstimatedMassIndex',        'LV structure/remodelling',   'continuous', None),
    ('LeftVentricleEstimatedMass',             'LV structure/remodelling',   'continuous', None),
    ('LeftVentriclePosteriorWallThickness',    'LV structure/remodelling',   'continuous', None),
    ('LeftVentricleEndSystolicDiameter',       'LV structure/remodelling',   'continuous', None),
    ('LeftVentricleEndDiastolicDiameter',      'LV structure/remodelling',   'continuous', None),
    ('LeftVentricleInterventricularSeptumThickness','LV structure/remodelling','continuous',None),
    ('LeftVentricleScoreIndex',                'LV structure/remodelling',   'continuous', None),
    ('LeftVentricleCavitySize',                'LV structure/remodelling',   'categorical','Normal'),
]
# Excluded: ECHO_SPAP (duplicate of PASP), LeftVentricleSystolicFunction (overlaps EF)
EXCLUDED_VARS = ['ECHO_SPAP','LeftVentricleSystolicFunction']

# Category collapse rules for adjusted models (3-group)
def collapse_mr_tr(val):
    if pd.isna(val) or str(val) in ('nan','No Value'): return np.nan
    v = str(val)
    if v == 'Trivial':  return 'None_Trivial'
    if v == 'Mild (I)': return 'Mild'
    return 'Mod_Severe'

def collapse_la(val):
    if pd.isna(val) or str(val) in ('nan','No Value'): return np.nan
    v = str(val)
    if v == 'Normal':         return 'Normal'
    if v == 'Mildly dilated': return 'Mildly_dilated'
    return 'Mod_Severely_dilated'

def collapse_lv_cavity(val):
    if pd.isna(val) or str(val) in ('nan','No Value'): return np.nan
    v = str(val)
    if v == 'Normal': return 'Normal'
    return 'Dilated'  # Mildly + Mod/Severely → Dilated

# collapsed_col → (collapse_func, reference_level)
CAT_COLLAPSE = {
    'LACavitySize'          : (collapse_la,         'Normal'),
    'MitralRegurgitation'   : (collapse_mr_tr,      'None_Trivial'),
    'TricuspidRegurgitation': (collapse_mr_tr,      'None_Trivial'),
    'LeftVentricleCavitySize': (collapse_lv_cavity, 'Normal'),
}

print('Variable registry loaded:', len(ECHO_REGISTRY), 'variables')
print('Excluded:', EXCLUDED_VARS)

In [ ]:
def resolve(cands):
    for p in cands:
        if os.path.exists(p): return p
    return cands[0]

def read_auto(path):
    if path.lower().endswith(('.xlsx','.xls')): return pd.read_excel(path)
    for enc in ['utf-8-sig','utf-8','latin-1']:
        try: return pd.read_csv(path, low_memory=False, encoding=enc)
        except UnicodeDecodeError: pass
    raise RuntimeError(f'Cannot read: {path}')

try:
    from google.colab import files
    print('Upload: stage2_analysis_ready.csv (required)')
    print('Optional: stage2_missingness_summary.csv, stage2_variable_dictionary.csv')
    files.upload()
except Exception:
    print('Running locally.')

In [ ]:
df_raw = read_auto(resolve(['stage2_analysis_ready.csv','stage2_analysis_ready.xlsx']))
if 'patient number' in df_raw.columns and 'patient_id' not in df_raw.columns:
    df_raw = df_raw.rename(columns={'patient number':'patient_id'})
df_raw = df_raw.rename(columns={v:safe(v) for v in df_raw.columns})
df_raw = df_raw.loc[:, ~df_raw.columns.duplicated()].copy()

# Derive outcomes
tte = pd.to_numeric(df_raw['time_to_event_days'], errors='coerce')
df_raw['follow_up_years'] = tte / 365.25
df_raw['death_1y'] = df_raw.apply(
    lambda r: (1 if (r['event']==1 and tte[r.name]<=ONE_YEAR)
               else (0 if tte[r.name]>=ONE_YEAR else np.nan)), axis=1)
if 'm_f' in df_raw.columns:
    df_raw['sex_male'] = df_raw['m_f'].astype(str).str.lower().str.strip().map(
        {'m':1,'male':1,'f':0,'female':0})
if 'echo_to_dialysis_days' not in df_raw.columns:
    for cn in ['days_echo_to_dialysis','echo_to_dialysis']:
        if cn in df_raw.columns:
            df_raw['echo_to_dialysis_days'] = pd.to_numeric(df_raw[cn],errors='coerce').abs()
            break

# Apply collapse columns
for raw, (func, _) in CAT_COLLAPSE.items():
    sc = safe(raw)
    if sc in df_raw.columns:
        df_raw[sc + '_c'] = df_raw[sc].map(func)

df = df_raw.copy()
N  = len(df)
RENAL_S  = safe(RENAL_PRIMARY)
HOSP_S   = 'hosp_total' if 'hosp_total' in df.columns else 'hospitalization_count'
BASE_ALL = ['AgeAtFirstHFDate', 'sex_male', RENAL_S, 'echo_to_dialysis_days']
BASE     = [v for v in BASE_ALL if v in df.columns]

df_eval = df[df['death_1y'].notna()].copy()
df_hosp = df[df['follow_up_years'] > 0].copy()

print(f'N={N} | Mortality eval={len(df_eval)} | Deaths={int((df_eval["death_1y"]==1).sum())}')
print(f'Hospitalisation eval={len(df_hosp)} | Base covariates: {BASE}')

# ── Master variable table ─────────────────────────────────────
master_rows = []
for raw, domain, vtype, ref in ECHO_REGISTRY:
    sc = safe(raw)
    present = sc in df.columns
    miss_pct = round(effective_miss(sc, df[sc])*100,1) if present else np.nan
    master_rows.append({
        'variable_name':raw,'safe_name':sc,'domain':domain,
        'variable_type':vtype,'clinical_ref':ref if ref else 'continuous',
        'missing_pct':miss_pct,'n_valid':int((~effective_miss(sc,df[sc]).astype(bool)).sum()) if present else 0,
        'in_df':present,'included_in_broad_screen':present and (miss_pct < 20 if pd.notna(miss_pct) else False),
        'notes':''
    })

master_df = pd.DataFrame(master_rows)
# Fix the n_valid calculation
for i, row in master_df.iterrows():
    sc = row['safe_name']
    if sc in df.columns:
        t  = df[sc].astype(str)
        nv = int((~(df[sc].isna()|t.eq('No Value')|t.eq('nan'))).sum())
        mr = round((df[sc].isna()|t.eq('No Value')|t.eq('nan')).mean()*100,1)
        master_df.at[i,'n_valid']      = nv
        master_df.at[i,'missing_pct']  = mr
        master_df.at[i,'included_in_broad_screen'] = mr < 20

ECHO_VARS = [r['safe_name'] for _, r in master_df.iterrows()
             if r['included_in_broad_screen']]
print(f'\nIncluded in broad screen: {len(ECHO_VARS)} variables')
display(master_df[['variable_name','domain','variable_type','missing_pct','included_in_broad_screen']])
master_df.to_csv('broad_echo_screen_master_table.csv', index=False, encoding='utf-8-sig')
print('Saved: broad_echo_screen_master_table.csv')

---
## 2. Univariable mortality screening <a id='s2'></a>

In [ ]:
def logit_univ_continuous(df_m, col, label, domain):
    """Univariable logistic for continuous: per raw unit + per 1 SD."""
    df_m = df_m.loc[:, ~df_m.columns.duplicated()].copy()
    sub  = df_m[['death_1y', col]].dropna().copy()
    s    = pd.to_numeric(get_series(sub, col), errors='coerce')
    sd   = s.std()
    rows = []
    for rep, scale in [('per raw unit', 1.0), ('per 1 SD', sd)]:
        if pd.isna(scale) or scale == 0:
            rows.append({'variable':label,'domain':domain,'type':'continuous',
                         'representation':rep,'reference':'','N':len(sub),
                         'OR':np.nan,'CI_lo':np.nan,'CI_hi':np.nan,'p_value':np.nan,
                         'estimable':False,'unstable':False,'reason':'SD=0','notes':f'1SD={round(sd,2)}'})
            continue
        sub2 = sub.copy()
        sub2['_x'] = s / scale
        try:
            mod = smf.logit('death_1y ~ _x', data=sub2).fit(disp=False, maxiter=200)
            c   = mod.params['_x']; ci = mod.conf_int().loc['_x']; p = mod.pvalues['_x']
            lo, hi = np.exp(ci[0]), np.exp(ci[1])
            ci_r = hi/lo if lo>0 else np.inf
            unst = (abs(c)>COEF_T) or (ci_r>CI_W)
            rows.append({'variable':label,'domain':domain,'type':'continuous',
                         'representation':rep,'reference':'','N':len(sub),
                         'OR':round(np.exp(c),3),'CI_lo':round(lo,3),'CI_hi':round(hi,3),
                         'p_value':round(p,4),'estimable':True,'unstable':unst,
                         'reason':'extreme_CI' if unst else '',
                         'notes':f'1SD={round(sd,2)}'})
        except Exception as e:
            rows.append({'variable':label,'domain':domain,'type':'continuous',
                         'representation':rep,'reference':'','N':len(sub),
                         'OR':np.nan,'CI_lo':np.nan,'CI_hi':np.nan,'p_value':np.nan,
                         'estimable':False,'unstable':True,
                         'reason':str(e)[:60],'notes':f'1SD={round(sd,2)}'})
    return rows

def logit_univ_categorical(df_m, col, ref, label, domain):
    """Univariable logistic for categorical: all levels vs clinical reference."""
    df_m = df_m.loc[:, ~df_m.columns.duplicated()].copy()
    sub  = df_m[['death_1y', col]].copy()
    sub  = sub[~(sub[col].isna() | sub[col].astype(str).isin(['No Value','nan']))].copy()
    rows = []
    cats = [c for c in sub[col].unique() if str(c) != ref]
    sparse = [c for c in sub[col].unique() if sub[col].eq(c).sum() < 5]
    if sparse:
        rows.append({'variable':label,'domain':domain,'type':'categorical',
                     'representation':f'all levels vs {ref}','reference':ref,'N':len(sub),
                     'OR':np.nan,'CI_lo':np.nan,'CI_hi':np.nan,'p_value':np.nan,
                     'estimable':False,'unstable':False,
                     'reason':f'sparse_categories:{sparse}','notes':''})
        return rows
    try:
        mod = smf.logit(f"death_1y ~ C({col}, Treatment('{ref}'))", data=sub).fit(disp=False, maxiter=200)
        for k in [kk for kk in mod.params.index if col in kk and kk != 'Intercept']:
            lv   = k.split('[T.')[-1].rstrip(']')
            c    = mod.params[k]; ci = mod.conf_int().loc[k]; p = mod.pvalues[k]
            lo, hi = np.exp(ci[0]), np.exp(ci[1])
            ci_r = hi/lo if lo>0 else np.inf
            rows.append({'variable':label,'domain':domain,'type':'categorical',
                         'representation':f'{lv} vs {ref}','reference':ref,'N':len(sub),
                         'OR':round(np.exp(c),3),'CI_lo':round(lo,3),'CI_hi':round(hi,3),
                         'p_value':round(p,4),
                         'estimable':True,'unstable':(abs(c)>COEF_T or ci_r>CI_W),
                         'reason':'extreme_CI' if (abs(c)>COEF_T or ci_r>CI_W) else '',
                         'notes':f'ref={ref}(clinical)'})
    except Exception as e:
        rows.append({'variable':label,'domain':domain,'type':'categorical',
                     'representation':f'all levels vs {ref}','reference':ref,'N':len(sub),
                     'OR':np.nan,'CI_lo':np.nan,'CI_hi':np.nan,'p_value':np.nan,
                     'estimable':False,'unstable':True,
                     'reason':str(e)[:60],'notes':''})
    return rows

# ── Run univariable mortality ─────────────────────────────────
univ_mort_rows = []
for raw, domain, vtype, ref in ECHO_REGISTRY:
    sc = safe(raw)
    if sc not in df_eval.columns: continue
    if vtype == 'continuous':
        univ_mort_rows += logit_univ_continuous(df_eval, sc, raw, domain)
    else:
        univ_mort_rows += logit_univ_categorical(df_eval, sc, ref, raw, domain)

univ_mort_df = pd.DataFrame(univ_mort_rows)
print('── Univariable mortality (per 1 SD, p-sorted) ──')
display(univ_mort_df[univ_mort_df['representation'].str.contains('1 SD|Mod_Severe|Mod_Severely')]
        [['variable','domain','representation','N','OR','CI_lo','CI_hi','p_value','estimable','unstable']]
        .sort_values('p_value'))
univ_mort_df.to_csv('broad_echo_univariate_mortality.csv', index=False, encoding='utf-8-sig')
print('Saved: broad_echo_univariate_mortality.csv')

# Promising: any representation with p<0.10
promising_mort = list(univ_mort_df[
    (univ_mort_df['p_value'] < P_SCREEN) & (univ_mort_df['estimable']==True)
]['variable'].unique())
print(f'Promising for mortality (p<{P_SCREEN}): {promising_mort}')

---
## 3. Univariable hospitalisation screening <a id='s3'></a>

In [ ]:
def od_ratio(df_m, col):
    s = pd.to_numeric(get_series(df_m[df_m['follow_up_years']>0], col), errors='coerce').dropna()
    if len(s) < 10: return np.inf
    v = float(s.var()); m = float(s.mean())
    return v/m if m > 0 else np.inf

def nb_univ_continuous(df_m, col, label, domain):
    df_m = df_m.loc[:, ~df_m.columns.duplicated()].copy()
    sub  = df_m[[HOSP_S, col, 'follow_up_years']].dropna().copy()
    sub  = sub[sub['follow_up_years'] > 0].copy()
    sub['log_fu'] = np.log(sub['follow_up_years'])
    s    = pd.to_numeric(get_series(sub, col), errors='coerce')
    sd   = s.std()
    rows = []
    for rep, scale in [('per raw unit', 1.0), ('per 1 SD', sd)]:
        if pd.isna(scale) or scale == 0:
            rows.append({'variable':label,'domain':domain,'type':'continuous',
                         'model_type':'GLM_NB','representation':rep,
                         'reference':'','N':len(sub),
                         'IRR':np.nan,'CI_lo':np.nan,'CI_hi':np.nan,'p_value':np.nan,
                         'estimable':False,'unstable':False,'converged':False,
                         'reason':'SD=0','notes':f'1SD={round(sd,2)}'})
            continue
        sub2 = sub.copy()
        sub2['_x'] = s / scale
        try:
            import warnings as _w
            with _w.catch_warnings(record=True) as _wl:
                _w.simplefilter('always')
                mod = smf.glm(f'{HOSP_S} ~ _x', data=sub2,
                              family=sm.families.NegativeBinomial(alpha=1.0),
                              offset=sub2['log_fu']).fit(disp=False)
            wm   = '; '.join(str(w.message) for w in _wl)[:100] if _wl else ''
            conv = bool(getattr(mod,'converged',True))
            c    = mod.params['_x']; ci = mod.conf_int().loc['_x']; p = mod.pvalues['_x']
            lo, hi = np.exp(ci[0]), np.exp(ci[1])
            ci_r = hi/lo if lo>0 else np.inf
            unst = (abs(c)>COEF_T) or (ci_r>CI_W) or (not conv)
            rows.append({'variable':label,'domain':domain,'type':'continuous',
                         'model_type':'GLM_NB','representation':rep,
                         'reference':'','N':len(sub),
                         'IRR':round(np.exp(c),3),'CI_lo':round(lo,3),'CI_hi':round(hi,3),
                         'p_value':round(p,4),'estimable':True,'unstable':unst,
                         'converged':conv,'reason':'extreme_CI' if unst else '',
                         'notes':f'1SD={round(sd,2)} warn:{wm[:50]}'})
        except Exception as e:
            rows.append({'variable':label,'domain':domain,'type':'continuous',
                         'model_type':'GLM_NB','representation':rep,
                         'reference':'','N':len(sub),
                         'IRR':np.nan,'CI_lo':np.nan,'CI_hi':np.nan,'p_value':np.nan,
                         'estimable':False,'unstable':True,'converged':False,
                         'reason':str(e)[:60],'notes':f'1SD={round(sd,2)}'})
    return rows

def nb_univ_categorical(df_m, col, ref, label, domain):
    df_m = df_m.loc[:, ~df_m.columns.duplicated()].copy()
    sub  = df_m[[HOSP_S, col, 'follow_up_years']].copy()
    sub  = sub[~(sub[col].isna()|sub[col].astype(str).isin(['No Value','nan']))]
    sub  = sub[sub['follow_up_years']>0].copy()
    sub['log_fu'] = np.log(sub['follow_up_years'])
    sparse = [c for c in sub[col].unique() if sub[col].eq(c).sum() < 5]
    rows = []
    if sparse:
        rows.append({'variable':label,'domain':domain,'type':'categorical',
                     'model_type':'GLM_NB','representation':f'all vs {ref}',
                     'reference':ref,'N':len(sub),
                     'IRR':np.nan,'CI_lo':np.nan,'CI_hi':np.nan,'p_value':np.nan,
                     'estimable':False,'unstable':False,'converged':False,
                     'reason':f'sparse:{sparse}','notes':''})
        return rows
    try:
        mod = smf.glm(f"{HOSP_S} ~ C({col}, Treatment('{ref}'))", data=sub,
                      family=sm.families.NegativeBinomial(alpha=1.0),
                      offset=sub['log_fu']).fit(disp=False)
        conv = bool(getattr(mod,'converged',True))
        for k in [kk for kk in mod.params.index if col in kk and kk != 'Intercept']:
            lv   = k.split('[T.')[-1].rstrip(']')
            c    = mod.params[k]; ci = mod.conf_int().loc[k]; p = mod.pvalues[k]
            lo, hi = np.exp(ci[0]), np.exp(ci[1])
            rows.append({'variable':label,'domain':domain,'type':'categorical',
                         'model_type':'GLM_NB','representation':f'{lv} vs {ref}',
                         'reference':ref,'N':len(sub),
                         'IRR':round(np.exp(c),3),'CI_lo':round(lo,3),'CI_hi':round(hi,3),
                         'p_value':round(p,4),'estimable':True,'unstable':False,
                         'converged':conv,'reason':'','notes':f'ref={ref}(clinical)'})
    except Exception as e:
        rows.append({'variable':label,'domain':domain,'type':'categorical',
                     'model_type':'GLM_NB','representation':f'all vs {ref}',
                     'reference':ref,'N':len(sub),
                     'IRR':np.nan,'CI_lo':np.nan,'CI_hi':np.nan,'p_value':np.nan,
                     'estimable':False,'unstable':True,'converged':False,
                     'reason':str(e)[:60],'notes':''})
    return rows

# ── Run univariable hosp ──────────────────────────────────────
univ_hosp_rows = []
for raw, domain, vtype, ref in ECHO_REGISTRY:
    sc = safe(raw)
    if sc not in df_hosp.columns: continue
    if vtype == 'continuous':
        univ_hosp_rows += nb_univ_continuous(df_hosp, sc, raw, domain)
    else:
        univ_hosp_rows += nb_univ_categorical(df_hosp, sc, ref, raw, domain)

univ_hosp_df = pd.DataFrame(univ_hosp_rows)
print('── Univariable hospitalisation (per 1 SD, p-sorted) ──')
display(univ_hosp_df[univ_hosp_df['representation'].str.contains('1 SD|Mod_Severe|Mod_Severely')]
        [['variable','domain','representation','N','IRR','CI_lo','CI_hi','p_value','estimable']]
        .sort_values('p_value'))
univ_hosp_df.to_csv('broad_echo_univariate_hospitalization.csv', index=False, encoding='utf-8-sig')
print('Saved: broad_echo_univariate_hospitalization.csv')

promising_hosp = list(univ_hosp_df[
    (univ_hosp_df['p_value'] < P_SCREEN) & (univ_hosp_df['estimable']==True)
]['variable'].unique())
print(f'Promising for hosp (p<{P_SCREEN}): {promising_hosp}')

---
## 4. Adjusted mortality models <a id='s4'></a>

In [ ]:
# Adjusted models run only for promising vars (p<0.10 in univariable)
# Running 19 adjusted models would be excessive; restricting to promising vars
# keeps the analysis focused and stable.

def adj_logit(df_m, col, vtype, ref_for_adj, label, domain):
    df_m = df_m.loc[:, ~df_m.columns.duplicated()].copy()
    # For categorical: use collapsed _c version
    is_cat = (vtype == 'categorical')
    if is_cat:
        func, ref = CAT_COLLAPSE.get(col, (None, ref_for_adj))
        ev = col + '_c'
        if ev not in df_m.columns:
            return [{'variable':label,'domain':domain,'type':vtype,'representation':'adjusted_cat',
                     'reference':ref,'N':0,'OR':np.nan,'CI_lo':np.nan,'CI_hi':np.nan,
                     'p_value':np.nan,'estimable':False,'unstable':False,
                     'reason':'collapsed_col_missing','notes':''}]
    else:
        ev, ref = col, None

    needed = ['death_1y', ev] + BASE
    sub = df_m[[c for c in needed if c in df_m.columns]].dropna().copy()
    n   = len(sub)
    if n < 30:
        return [{'variable':label,'domain':domain,'type':vtype,'representation':'adjusted',
                 'reference':ref or '','N':n,'OR':np.nan,'CI_lo':np.nan,'CI_hi':np.nan,
                 'p_value':np.nan,'estimable':False,'unstable':False,
                 'reason':f'N<30','notes':''}]
    if is_cat:
        sparse = [c for c,cnt in sub[ev].value_counts().items() if cnt < 5]
        if sparse:
            return [{'variable':label,'domain':domain,'type':vtype,'representation':'adjusted_cat',
                     'reference':ref,'N':n,'OR':np.nan,'CI_lo':np.nan,'CI_hi':np.nan,
                     'p_value':np.nan,'estimable':False,'unstable':False,
                     'reason':f'sparse_cats:{sparse}','notes':''}]

    avail_adj = [v for v in BASE if v in sub.columns and pd.to_numeric(sub[v],errors='coerce').std()>0]
    if is_cat:
        formula = f"death_1y ~ C({ev}, Treatment('{ref}')) + " + ' + '.join(avail_adj)
    else:
        formula = f'death_1y ~ {ev} + ' + ' + '.join(avail_adj)
    rows = []
    try:
        mod = smf.logit(formula, data=sub).fit(disp=False, maxiter=200)
        keys = [k for k in mod.params.index if ev in k and k != 'Intercept'] if is_cat else [ev]
        for k in keys:
            lv = k.split('[T.')[-1].rstrip(']') if is_cat else 'per unit'
            c  = mod.params[k]; ci = mod.conf_int().loc[k]; p = mod.pvalues[k]
            lo, hi = np.exp(ci[0]), np.exp(ci[1])
            ci_r = hi/lo if lo>0 else np.inf
            rows.append({'variable':label,'domain':domain,'type':vtype,
                         'representation':f'{lv} vs {ref}' if is_cat else 'per unit adjusted',
                         'reference':ref or '','N':n,
                         'OR':round(np.exp(c),3),'CI_lo':round(lo,3),'CI_hi':round(hi,3),
                         'p_value':round(p,4),'estimable':True,
                         'unstable':(abs(c)>COEF_T or ci_r>CI_W),
                         'reason':'extreme_CI' if (abs(c)>COEF_T or ci_r>CI_W) else '',
                         'notes':''})
    except Exception as e:
        rows.append({'variable':label,'domain':domain,'type':vtype,'representation':'adjusted',
                     'reference':ref or '','N':n,'OR':np.nan,'CI_lo':np.nan,'CI_hi':np.nan,
                     'p_value':np.nan,'estimable':False,'unstable':True,
                     'reason':str(e)[:60],'notes':''})
    return rows

adj_mort_rows = []
ADJUSTED_VARS_MORT = list(dict.fromkeys(promising_mort + ['LV_EF']))
print(f'Adjusted mortality models for: {ADJUSTED_VARS_MORT}')

for raw, domain, vtype, ref in ECHO_REGISTRY:
    if raw not in ADJUSTED_VARS_MORT: continue
    sc = safe(raw)
    if sc not in df_eval.columns: continue
    adj_mort_rows += adj_logit(df_eval, sc, vtype, ref, raw, domain)

adj_mort_df = pd.DataFrame(adj_mort_rows)
display(adj_mort_df[['variable','domain','representation','N','OR','CI_lo','CI_hi','p_value','estimable','unstable']])
adj_mort_df.to_csv('broad_echo_adjusted_mortality.csv', index=False, encoding='utf-8-sig')
print('Saved: broad_echo_adjusted_mortality.csv')

---
## 5. Adjusted hospitalisation models <a id='s5'></a>

In [ ]:
def adj_nb(df_m, col, vtype, ref_for_adj, label, domain):
    df_m = df_m.loc[:, ~df_m.columns.duplicated()].copy()
    is_cat = (vtype == 'categorical')
    if is_cat:
        _, ref = CAT_COLLAPSE.get(col, (None, ref_for_adj))
        ev = col + '_c'
        if ev not in df_m.columns:
            return [{'variable':label,'domain':domain,'type':vtype,'representation':'adjusted_cat',
                     'reference':ref,'N':0,'IRR':np.nan,'CI_lo':np.nan,'CI_hi':np.nan,
                     'p_value':np.nan,'estimable':False,'unstable':False,'converged':False,
                     'reason':'collapsed_col_missing','notes':''}]
    else:
        ev, ref = col, None

    needed = [HOSP_S, ev, 'follow_up_years'] + BASE
    sub = df_m[[c for c in needed if c in df_m.columns]].dropna().copy()
    sub = sub[sub['follow_up_years']>0].copy()
    sub = sub.loc[:, ~sub.columns.duplicated()].copy()
    sub['log_fu'] = np.log(sub['follow_up_years'])
    n = len(sub)
    if n < 30:
        return [{'variable':label,'domain':domain,'type':vtype,'representation':'adjusted',
                 'reference':ref or '','N':n,'IRR':np.nan,'CI_lo':np.nan,'CI_hi':np.nan,
                 'p_value':np.nan,'estimable':False,'unstable':False,'converged':False,
                 'reason':'N<30','notes':''}]
    if is_cat:
        sparse = [c for c,cnt in sub[ev].value_counts().items() if cnt < 5]
        if sparse:
            return [{'variable':label,'domain':domain,'type':vtype,'representation':'adjusted_cat',
                     'reference':ref,'N':n,'IRR':np.nan,'CI_lo':np.nan,'CI_hi':np.nan,
                     'p_value':np.nan,'estimable':False,'unstable':False,'converged':False,
                     'reason':f'sparse_cats:{sparse}','notes':''}]

    avail_adj = [v for v in BASE if v in sub.columns and pd.to_numeric(sub[v],errors='coerce').std()>0]
    formula = (f"{HOSP_S} ~ C({ev}, Treatment('{ref}')) + " + ' + '.join(avail_adj)
               if is_cat else f'{HOSP_S} ~ {ev} + ' + ' + '.join(avail_adj))
    rows = []
    try:
        import warnings as _w
        with _w.catch_warnings(record=True):
            _w.simplefilter('always')
            mod = smf.glm(formula, data=sub,
                          family=sm.families.NegativeBinomial(alpha=1.0),
                          offset=sub['log_fu']).fit(disp=False)
        conv = bool(getattr(mod,'converged',True))
        keys = [k for k in mod.params.index if ev in k and k != 'Intercept'] if is_cat else [ev]
        for k in keys:
            lv = k.split('[T.')[-1].rstrip(']') if is_cat else 'per unit'
            c  = mod.params[k]; ci = mod.conf_int().loc[k]; p = mod.pvalues[k]
            lo, hi = np.exp(ci[0]), np.exp(ci[1])
            ci_r = hi/lo if lo>0 else np.inf
            rows.append({'variable':label,'domain':domain,'type':vtype,
                         'representation':f'{lv} vs {ref}' if is_cat else 'per unit adjusted',
                         'reference':ref or '','N':n,
                         'IRR':round(np.exp(c),3),'CI_lo':round(lo,3),'CI_hi':round(hi,3),
                         'p_value':round(p,4),'estimable':True,'converged':conv,
                         'unstable':(abs(c)>COEF_T or ci_r>CI_W or not conv),
                         'reason':'extreme_CI_or_no_conv' if (abs(c)>COEF_T or ci_r>CI_W or not conv) else '',
                         'notes':''})
    except Exception as e:
        rows.append({'variable':label,'domain':domain,'type':vtype,'representation':'adjusted',
                     'reference':ref or '','N':n,'IRR':np.nan,'CI_lo':np.nan,'CI_hi':np.nan,
                     'p_value':np.nan,'estimable':False,'unstable':True,'converged':False,
                     'reason':str(e)[:60],'notes':''})
    return rows

adj_hosp_rows = []
ADJUSTED_VARS_HOSP = list(dict.fromkeys(promising_hosp + ['LV_EF']))
print(f'Adjusted hosp models for: {ADJUSTED_VARS_HOSP}')

for raw, domain, vtype, ref in ECHO_REGISTRY:
    if raw not in ADJUSTED_VARS_HOSP: continue
    sc = safe(raw)
    if sc not in df_hosp.columns: continue
    adj_hosp_rows += adj_nb(df_hosp, sc, vtype, ref, raw, domain)

adj_hosp_df = pd.DataFrame(adj_hosp_rows)
display(adj_hosp_df[['variable','domain','representation','N','IRR','CI_lo','CI_hi','p_value','estimable','unstable']])
adj_hosp_df.to_csv('broad_echo_adjusted_hospitalization.csv', index=False, encoding='utf-8-sig')
print('Saved: broad_echo_adjusted_hospitalization.csv')

---
## 6. Beyond-EF: matched-sample incremental models <a id='s6'></a>

In [ ]:
# For each promising var: matched subset → refit M1 (base+EF) AND M2 (base+EF+var)
# LRT and ΔAIC valid ONLY when N is identical.

def fit_logit_m(df_m, formula, label):
    df_m = df_m.loc[:, ~df_m.columns.duplicated()].copy()
    sub  = df_m.dropna().copy()
    if len(sub) < 30:
        return {'label':label,'N':len(sub),'ll':np.nan,'AIC':np.nan,'mod':None}
    try:
        mod = smf.logit(formula, data=sub).fit(disp=False, maxiter=200)
        return {'label':label,'N':int(mod.nobs),'ll':round(mod.llf,3),
                'AIC':round(mod.aic,3),'mod':mod}
    except:
        return {'label':label,'N':len(sub),'ll':np.nan,'AIC':np.nan,'mod':None}

def fit_nb_m(df_m, formula, label):
    df_m = df_m.loc[:, ~df_m.columns.duplicated()].copy()
    sub  = df_m.dropna().copy()
    sub  = sub[sub['follow_up_years']>0].copy()
    sub['log_fu'] = np.log(sub['follow_up_years'])
    if len(sub) < 30:
        return {'label':label,'N':len(sub),'ll':np.nan,'AIC':np.nan,'mod':None}
    try:
        import warnings as _w
        with _w.catch_warnings(record=True): _w.simplefilter('always')
        mod = smf.glm(formula, data=sub,
                      family=sm.families.NegativeBinomial(alpha=1.0),
                      offset=sub['log_fu']).fit(disp=False)
        return {'label':label,'N':int(mod.nobs),'ll':round(mod.llf,3),
                'AIC':round(mod.aic,3),'mod':mod}
    except:
        return {'label':label,'N':len(sub),'ll':np.nan,'AIC':np.nan,'mod':None}

def lrt_matched(r_null, r_full, df_diff):
    if r_null['mod'] is None or r_full['mod'] is None: return np.nan, np.nan
    chi2 = 2*(r_full['ll']-r_null['ll'])
    p    = stats.chi2.sf(chi2, df_diff)
    return round(chi2,3), round(p,4)

f_base_ef = 'death_1y ~ ' + ' + '.join(BASE) + ' + LV_EF'
f_base_ef_h = HOSP_S + ' ~ ' + ' + '.join(BASE) + ' + LV_EF'

bef_mort_rows = []
bef_hosp_rows = []

BEYOND_EF_VARS = list(dict.fromkeys(promising_mort + promising_hosp))
print(f'Beyond-EF analysis for: {BEYOND_EF_VARS}')

for raw, domain, vtype, ref in ECHO_REGISTRY:
    if raw == 'LV_EF': continue
    sc = safe(raw)
    is_cat = (vtype == 'categorical')

    # ── Mortality beyond EF ───────────────────────────────────
    if raw in BEYOND_EF_VARS and sc in df_eval.columns:
        if is_cat:
            _, cref = CAT_COLLAPSE.get(sc, (None, ref))
            ev_c = sc + '_c'
            pred = f"C({ev_c}, Treatment('{cref}'))"
            df_diff = 2  # 2 dummy params for 3-group collapse
            all_cols = ['death_1y', ev_c, 'LV_EF'] + BASE
        else:
            pred = sc; df_diff = 1
            all_cols = ['death_1y', sc, 'LV_EF'] + BASE

        avail_cols = [c for c in dict.fromkeys(all_cols) if c in df_eval.columns]
        df_matched = df_eval[avail_cols].dropna().copy()
        n_matched  = len(df_matched)
        r_null = fit_logit_m(df_matched, f_base_ef, 'M1_refit')
        r_full = fit_logit_m(df_matched, f_base_ef + f' + {pred}', 'M2_full')
        chi2, p = lrt_matched(r_null, r_full, df_diff)
        bef_mort_rows.append({
            'variable':raw,'domain':domain,
            'subset_N':n_matched,'M1_refit_subset_N':r_null['N'],
            'comparison_valid':(r_null['N']==r_full['N'] and r_null['mod'] is not None),
            'LRT_chi2':chi2,'LRT_p':p,
            'delta_AIC':round(r_full['AIC']-r_null['AIC'],2) if not np.isnan(r_full.get('AIC',np.nan)) else np.nan,
            'beyond_EF_yes_no':'YES' if(not np.isnan(p) and p<0.10) else 'NO',
        })

    # ── Hospitalisation beyond EF ─────────────────────────────
    if raw in BEYOND_EF_VARS and sc in df_hosp.columns:
        if is_cat:
            _, cref = CAT_COLLAPSE.get(sc, (None, ref))
            ev_c = sc + '_c'; pred = f"C({ev_c}, Treatment('{cref}'))"
            df_diff_h = 2
            all_cols_h = [HOSP_S, ev_c, 'follow_up_years', 'LV_EF'] + BASE
        else:
            pred = sc; df_diff_h = 1
            all_cols_h = [HOSP_S, sc, 'follow_up_years', 'LV_EF'] + BASE

        avail_h = [c for c in dict.fromkeys(all_cols_h) if c in df_hosp.columns]
        df_h_matched = df_hosp[avail_h].dropna().copy()
        df_h_matched = df_h_matched[df_h_matched['follow_up_years']>0].copy()
        n_hm = len(df_h_matched)
        r_null_h = fit_nb_m(df_h_matched, f_base_ef_h, 'H1_refit')
        r_full_h = fit_nb_m(df_h_matched, f_base_ef_h + f' + {pred}', 'H2_full')
        chi2_h, p_h = lrt_matched(r_null_h, r_full_h, df_diff_h)
        bef_hosp_rows.append({
            'variable':raw,'domain':domain,
            'subset_N':n_hm,'M1_refit_subset_N':r_null_h['N'],
            'comparison_valid':(r_null_h['N']==r_full_h['N'] and r_null_h['mod'] is not None),
            'LRT_chi2':chi2_h,'LRT_p':p_h,
            'delta_AIC':round(r_full_h['AIC']-r_null_h['AIC'],2) if not np.isnan(r_full_h.get('AIC',np.nan)) else np.nan,
            'beyond_EF_yes_no':'YES' if(not np.isnan(p_h) and p_h<0.10) else 'NO',
        })

bef_mort_df = pd.DataFrame(bef_mort_rows)
bef_hosp_df = pd.DataFrame(bef_hosp_rows)
print('── Beyond EF — mortality ──')
display(bef_mort_df)
print('── Beyond EF — hospitalisation ──')
display(bef_hosp_df)
bef_mort_df.to_csv('broad_echo_beyond_EF_mortality.csv', index=False, encoding='utf-8-sig')
bef_hosp_df.to_csv('broad_echo_beyond_EF_hospitalization.csv', index=False, encoding='utf-8-sig')
print('Saved: beyond_EF CSVs')

---
## 7. Overlap analysis <a id='s7'></a>

In [ ]:
# Clinically motivated pairs only — not all-vs-all
# 8A: continuous-continuous (Spearman)
# 8B: categorical-categorical (Cramér's V)
# 8C: continuous vs categorical (Kruskal-Wallis)

OVERLAP_PAIRS = [
    # (var1, var2, domain, method_note)
    # Diastolic metrics
    ('TissueDopplerEERatioSeptal',  'TissueDopplerEERatioLateral',   'diastolic/filling',         'cont-cont'),
    ('TissueDopplerEERatioSeptal',  'MitralInflowPeakEWave',         'diastolic/filling',         'cont-cont'),
    ('TissueDopplerEVelositySeptal','TissueDopplerEVelosityLateral', 'diastolic/filling',         'cont-cont'),
    ('TissueDopplerEERatioSeptal',  'TissueDopplerEVelositySeptal',  'diastolic/filling',         'cont-cont'),
    ('TissueDopplerEERatioLateral', 'TissueDopplerEVelosityLateral', 'diastolic/filling',         'cont-cont'),
    ('MitralInflowPeakEWave',       'LACavitySize',                  'diastolic/filling',         'cont-cat'),
    ('TissueDopplerEERatioSeptal',  'LACavitySize',                  'diastolic/filling',         'cont-cat'),
    # LV structure
    ('LeftVentricleEstimatedMass',  'LeftVentricleEstimatedMassIndex','LV structure/remodelling', 'cont-cont'),
    ('LeftVentricleEndDiastolicDiameter','LeftVentricleEndSystolicDiameter','LV structure/remodelling','cont-cont'),
    ('LeftVentricleEndDiastolicDiameter','LeftVentricleCavitySize',  'LV structure/remodelling',  'cont-cat'),
    ('LeftVentricleInterventricularSeptumThickness','LeftVentriclePosteriorWallThickness','LV structure/remodelling','cont-cont'),
    ('LeftVentricleScoreIndex',     'LeftVentricleEstimatedMassIndex','LV structure/remodelling', 'cont-cont'),
    # Valvular / pulmonary
    ('MitralRegurgitation',         'TricuspidRegurgitation',        'valvular',                  'cat-cat'),
    ('TricuspidRegurgitation',      'EstimatedSysPAPressure',        'pulmonary/right-sided load','cat-cont'),
    ('EstimatedSysPAPressure',      'TissueDopplerEERatioSeptal',    'mixed',                     'cont-cont'),
    # EF vs structure
    ('LV_EF',                       'LeftVentricleEstimatedMassIndex','mixed',                    'cont-cont'),
    ('LV_EF',                       'LeftVentricleEndSystolicDiameter','mixed',                   'cont-cont'),
]

overlap_rows = []
from scipy.stats import spearmanr, kruskal

def strength_flag(r_or_v):
    a = abs(r_or_v)
    if a >= 0.50: return 'high overlap'
    if a >= 0.30: return 'moderate overlap'
    return 'low overlap'

for v1, v2, dom, meth in OVERLAP_PAIRS:
    s1, s2 = safe(v1), safe(v2)
    if s1 not in df.columns or s2 not in df.columns: continue

    if meth == 'cont-cont':
        c1 = pd.to_numeric(get_series(df[[s1]], s1), errors='coerce')
        c2 = pd.to_numeric(get_series(df[[s2]], s2), errors='coerce')
        valid = c1.notna() & c2.notna()
        if valid.sum() < 10: continue
        r, p = spearmanr(c1[valid], c2[valid])
        overlap_rows.append({'variable_1':v1,'variable_2':v2,'domain':dom,
                              'method':'Spearman r','N_pairwise':int(valid.sum()),
                              'association_measure':round(r,3),'p_value':round(p,4),
                              'strength_flag':strength_flag(r),
                              'interpretation':f'r={round(r,3)} ({strength_flag(r)})'})

    elif meth == 'cat-cat':
        ct = pd.crosstab(df[s1].astype(str), df[s2].astype(str))
        if ct.shape[0] < 2 or ct.shape[1] < 2: continue
        chi2, p_c, _, _ = stats.chi2_contingency(ct)
        cv = np.sqrt(chi2 / (ct.values.sum() * (min(ct.shape)-1)))
        overlap_rows.append({'variable_1':v1,'variable_2':v2,'domain':dom,
                              'method':"Cramér's V",'N_pairwise':int(ct.values.sum()),
                              'association_measure':round(cv,3),'p_value':round(p_c,4),
                              'strength_flag':strength_flag(cv),
                              'interpretation':f"V={round(cv,3)} ({strength_flag(cv)})"})

    elif meth in ('cont-cat','cat-cont'):
        cont_col, cat_col = (s1,s2) if meth=='cont-cat' else (s2,s1)
        cc = pd.to_numeric(get_series(df[[cont_col]], cont_col), errors='coerce')
        cats_valid = df[cat_col].astype(str)
        groups = [cc[(cats_valid==c)&cc.notna()].values for c in cats_valid.unique() if c not in ('nan','No Value')]
        groups = [g for g in groups if len(g) >= 3]
        if len(groups) < 2: continue
        _, p_kw = kruskal(*groups)
        overlap_rows.append({'variable_1':v1,'variable_2':v2,'domain':dom,
                              'method':'Kruskal-Wallis','N_pairwise':int(sum(len(g) for g in groups)),
                              'association_measure':round(p_kw,4),'p_value':round(p_kw,4),
                              'strength_flag':'high overlap' if p_kw<0.01 else 'moderate overlap' if p_kw<0.10 else 'low overlap',
                              'interpretation':f'KW p={round(p_kw,4)}'})

overlap_df = pd.DataFrame(overlap_rows)
display(overlap_df[['variable_1','variable_2','method','association_measure','strength_flag']])
overlap_df.to_csv('broad_echo_overlap_summary.csv', index=False, encoding='utf-8-sig')
print('Saved: broad_echo_overlap_summary.csv')

---
## 8. Prioritisation summary <a id='s8'></a>

In [ ]:
DOMAIN_MAP = {r[0]:r[1] for r in ECHO_REGISTRY}

def signal_flag(var, df_univ, effect_col='OR', p_col='p_value'):
    """Returns YES/NO/borderline based on any representation with p<0.05/0.10."""
    sub = df_univ[(df_univ['variable']==var) & (df_univ['estimable']==True)]
    if not len(sub): return 'NO'
    pmin = sub[p_col].min()
    if pd.isna(pmin): return 'NO'
    if pmin < 0.05:   return 'YES (p<0.05)'
    if pmin < 0.10:   return 'borderline (p<0.10)'
    return 'NO'

def beyond_ef_flag(var, bef_df):
    if not len(bef_df): return 'N/A'
    sub = bef_df[bef_df['variable']==var]
    if not len(sub): return 'N/A'
    return str(sub.iloc[0].get('beyond_EF_yes_no','N/A'))

def stable_flag(var, df_list):
    for df_ in df_list:
        if not len(df_): continue
        sub = df_[df_['variable']==var]
        if not len(sub): continue
        if sub.get('unstable', pd.Series([False])).any(): return 'unstable'
    return 'stable'

def overlap_flag(var):
    sub = overlap_df[(overlap_df['variable_1']==var)|(overlap_df['variable_2']==var)]
    high = sub[sub['strength_flag']=='high overlap']
    if len(high):
        others = [r['variable_2'] if r['variable_1']==var else r['variable_1']
                  for _,r in high.iterrows()]
        return f'high overlap with: {others}'
    return 'no high overlap'

def recommend(var, mort_sig, hosp_sig, bef_m, bef_h, stab, ovlp):
    """Rule-based preliminary recommendation."""
    if stab == 'unstable':       return 'clinically interesting but unstable'
    has_signal = 'YES' in mort_sig or 'YES' in hosp_sig
    bef_yes    = bef_m == 'YES' or bef_h == 'YES'
    high_ovlp  = 'high overlap' in ovlp
    if has_signal and bef_yes and not high_ovlp:
        return 'strong candidate'
    if has_signal and bef_yes and high_ovlp:
        return 'redundant with stronger variable'
    if has_signal and not bef_yes:
        return 'supportive / secondary'
    return 'low priority / no clear signal'

prio_rows = []
for raw, domain, vtype, ref in ECHO_REGISTRY:
    sc       = safe(raw)
    miss_row = master_df[master_df['variable_name']==raw]
    miss_pct = float(miss_row['missing_pct'].iloc[0]) if len(miss_row) else np.nan
    mort_sig = signal_flag(raw, univ_mort_df)
    hosp_sig = signal_flag(raw, univ_hosp_df, effect_col='IRR')
    bef_m    = beyond_ef_flag(raw, bef_mort_df)
    bef_h    = beyond_ef_flag(raw, bef_hosp_df)
    stab     = stable_flag(raw, [univ_mort_df, univ_hosp_df, adj_mort_df, adj_hosp_df])
    ovlp     = overlap_flag(raw)
    rec      = recommend(raw, mort_sig, hosp_sig, bef_m, bef_h, stab, ovlp)
    prio_rows.append({
        'variable':raw,'domain':domain,'variable_type':vtype,
        'missing_pct':miss_pct,
        'mortality_signal':mort_sig,'hospitalization_signal':hosp_sig,
        'beyond_EF_mortality':bef_m,'beyond_EF_hospitalization':bef_h,
        'overlap_flag':ovlp,'stability_flag':stab,
        'preliminary_recommendation':rec,
    })

prio_df = pd.DataFrame(prio_rows).sort_values(
    ['preliminary_recommendation','domain'],
    key=lambda x: x.map({'strong candidate':0,'supportive / secondary':1,
                          'redundant with stronger variable':2,
                          'clinically interesting but unstable':3,
                          'low priority / no clear signal':4}).fillna(9)
    if x.name=='preliminary_recommendation' else x
)
display(prio_df[['variable','domain','mortality_signal','hospitalization_signal',
                  'beyond_EF_mortality','beyond_EF_hospitalization',
                  'overlap_flag','stability_flag','preliminary_recommendation']])
prio_df.to_csv('broad_echo_prioritization_summary.csv', index=False, encoding='utf-8-sig')
print('Saved: broad_echo_prioritization_summary.csv')

print('\nRecommendation counts:')
print(prio_df['preliminary_recommendation'].value_counts().to_string())

---
## 9. QA <a id='s9'></a>

In [ ]:
# ── QA checks ─────────────────────────────────────────────────
qa(not any(c.endswith('.1') for c in df.columns), 'No .1 duplicate columns')
qa('ECHO_SPAP' not in ECHO_VARS, 'ECHO_SPAP excluded per protocol')
qa('LeftVentricleSystolicFunction' not in ECHO_VARS,
   'LeftVentricleSystolicFunction excluded per protocol')
qa(len(ECHO_VARS) >= 15, f'At least 15 variables in broad set', 15, len(ECHO_VARS))

n_est_mort = int((univ_mort_df['estimable']==False).sum())
n_est_hosp = int((univ_hosp_df['estimable']==False).sum())
qa(n_est_mort < len(ECHO_VARS), f'Univ mort: not all unestiable', None, n_est_mort)
qa(n_est_hosp < len(ECHO_VARS), f'Univ hosp: not all unestiable', None, n_est_hosp)

n_matched_invalid = int(bef_mort_df.get('comparison_valid', pd.Series([])).eq(False).sum())
qa(n_matched_invalid == 0, 'All beyond-EF mortality comparisons valid',
   0, n_matched_invalid)

required_files = [
    'broad_echo_screen_master_table.csv',
    'broad_echo_univariate_mortality.csv',
    'broad_echo_univariate_hospitalization.csv',
    'broad_echo_adjusted_mortality.csv',
    'broad_echo_adjusted_hospitalization.csv',
    'broad_echo_beyond_EF_mortality.csv',
    'broad_echo_beyond_EF_hospitalization.csv',
    'broad_echo_overlap_summary.csv',
    'broad_echo_prioritization_summary.csv',
    'broad_echo_modeling_qa_log.csv',
]

print('\n' + '='*65)
print('Broad echo screening — QA log')
print('='*65)
for msg in QA_LOG: print(' ', msg)
print(f'Warnings: {sum(1 for m in QA_LOG if m.startswith("WARN"))}/{len(QA_LOG)}')
print('='*65)

pd.DataFrame([{'check':m.split('  ',1)[-1],
               'status':'OK' if m.startswith('OK') else 'WARN'}
              for m in QA_LOG]).to_csv('broad_echo_modeling_qa_log.csv',
                                        index=False, encoding='utf-8-sig')

print('\nRequired output files:')
for f in required_files:
    print(f'  {"OK    " if os.path.exists(f) else "MISSING"}  {f}')

try:
    from google.colab import files
    for f in required_files:
        if os.path.exists(f): files.download(f)
except ImportError:
    print('Not in Colab — files saved locally.')